# Practical PyTorch · Part 14 — Companion Notebook

A math-free look at the one idea behind every transformer — **attention** — and the block built around it. We only build and read shapes; no training.

## 1. The idea

A `Linear` layer has no sense of word order; a `Conv2d` only looks at nearby pixels. Language needs neither — it needs every word to look at every other word and decide what's relevant. That is **self-attention**.

In *"The animal didn't cross the street because it was too tired,"* attention lets the word **it** look at every other word and put a high weight on **animal**, a low one on **street**.

## 2. Build a transformer block you can print

A transformer block is just self-attention + a small feed-forward MLP, with residual connections and layer norm riding along. PyTorch ships one ready-made.

Start small enough to connect to the twelve-word toy from Part 13: size a block to those four-number word-vectors and push a six-word sequence through it.

In [ ]:
import torch
import torch.nn as nn

# six word-vectors, four numbers each — like the toy embeddings from Part 13
emb = torch.randn(1, 6, 4)     # [batch, words, vector] = [1, 6, 4]

block = nn.TransformerEncoderLayer(
    d_model=4,              # match the four-number word-vectors
    nhead=2,                # must divide d_model (4 can't run 8 heads)
    dim_feedforward=8,      # the little MLP's hidden size
    batch_first=True,
)
out = block(emb)
print(out.shape)               # torch.Size([1, 6, 4])

Same shape in, same shape out: six word-vectors go in, six come back, each one now blended with the others by attention. Now scale it to the sizes a real model uses and print the block to read its parts.

In [ ]:
block = nn.TransformerEncoderLayer(
    d_model=128,            # the size of each word's vector
    nhead=8,                # run attention 8 ways in parallel ("heads")
    dim_feedforward=512,    # the hidden size of the little MLP
    batch_first=True,
)
print(block)

Read the printout: `self_attn` (the new idea), two `Linear` layers (the feed-forward MLP), and a couple of `LayerNorm`s (supporting cast). Count its parameters with the same one-liner from Part 5.

In [ ]:
total = sum(p.numel() for p in block.parameters())
print(f"{total:,} parameters")   # 198,272 parameters

## 3. Encoder vs decoder

Same block, two ways to point it:

- **Encoders** let every word see the whole sentence (both directions) — good for understanding. **BERT** / **DistilBERT**.
- **Decoders** let each word see only the words before it — good for generating one word at a time. **GPT**.

Next post we open a real encoder, DistilBERT, and find nothing inside but the block you just built — stacked six times.